# Recherche d'information dans la presse française de 1914

## Récupération des données


In [ ]:
!wget -q https://github.com/shuuchuu/datasets/raw/refs/heads/main/ghsparse.sh -O /tmp/ghsparse.sh && bash /tmp/ghsparse.sh french-press-1914

In [ ]:
!ls french-press-1914/le_figaro/1914/

## Corpus

Ce corpus réunit les numéros de 1914 de deux quotidiens français, *L'Humanité* et *Le Figaro*, numérisés par la Bibliothèque nationale de France. Chaque numéro est un fichier JSON, au chemin `<journal>/1914/<AAAAMMJJ>/<AAAAMMJJ>.metadata.fulltext.json` : un dossier par journal, puis un dossier par date de parution, qui contient le fichier de ce numéro.

La cellule précédente liste les dossiers de date du *Figaro* parce qu'ils tiennent à l'écran : le corpus n'a que 24 numéros du *Figaro*, entre le 19 janvier et le 18 novembre, contre 360 de *L'Humanité*. Ce déséquilibre se retrouvera dans les résultats des recherches.

Le texte vient d'une reconnaissance optique de caractères (OCR) brute, jamais corrigée : `french-press-1914/README.md` prévient qu'elle fait beaucoup d'erreurs, et le premier numéro du *Figaro* annonce une confiance de 0,57 (champ `format`). Des mots mal lus, coupés ou collés se retrouvent donc dans les paragraphes.

Trois champs des fichiers JSON servent ici :

- `contentAsText`, une liste de chaînes de caractères, dans lesquelles les paragraphes sont séparés par des sauts de ligne (`\n`) ;
- `date`, la date de parution, dans une liste d'une seule chaîne (`contenu["date"][0]`) ;
- `title`, le nom du journal, de même (`contenu["title"][0]`).

La cellule suivante charge le corpus dans `donnees`, un tableau `numpy.ndarray` à 3 colonnes, avec une ligne par paragraphe de plus de 10 mots :

- `donnees[:, 0]`, le texte du paragraphe, mis en minuscules et sans ponctuation par `pretraiter_texte` ;
- `donnees[:, 1]`, le journal ;
- `donnees[:, 2]`, la date du numéro.

In [ ]:
import datetime
import json
import pathlib
import re

import numpy
import tqdm.notebook


def pretraiter_texte(texte: str) -> str:
  return re.sub(r"\W+", " ", texte).lower()


def charger_corpus(dossier: pathlib.Path) -> numpy.ndarray:
  lignes = []
  for numero in tqdm.notebook.tqdm(list(dossier.rglob("*.json"))):
    with open(numero, encoding="utf8") as fichier:
      contenu = json.load(fichier)
    journal = contenu["title"][0]
    date = datetime.date.fromisoformat(contenu["date"][0]).strftime("%d/%m/%Y")
    for bloc in contenu["contentAsText"]:
      for paragraphe in bloc.split("\n"):
        paragraphe = pretraiter_texte(paragraphe)
        if len(paragraphe.split()) > 10:
          lignes.append((paragraphe, journal, date))
  # dtype=object : un tableau de chaînes à largeur fixe réserverait à chaque case la
  # place du plus long paragraphe (8 434 caractères), soit 4,9 Go pour ce corpus
  return numpy.array(lignes, dtype=object)


donnees = charger_corpus(pathlib.Path("french-press-1914"))
print(donnees.shape)

## Représentation TF-IDF

Créez un `TfidfVectorizer` dans une variable `vectorizer`, puis la matrice `X` des paragraphes du corpus (`donnees[:, 0]`) transformés en vecteurs de poids TF-IDF : une ligne par paragraphe, une colonne par mot du vocabulaire. La suite du TP réutilise `vectorizer` et `X`.

Affichez ensuite la forme de `X` et son nombre de valeurs non nulles : quelle part des cases de la matrice est non nulle ? Qu'en déduire sur la façon de la stocker ?

**Documentation utile**

- [`sklearn.feature_extraction.text.TfidfVectorizer`](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html)
- [`scipy.sparse.csr_matrix`](https://docs.scipy.org/doc/scipy/reference/generated/scipy.sparse.csr_matrix.html)

In [ ]:
# Votre code ici

### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `sklearn.feature_extraction.text.TfidfVectorizer()` : les paramètres par défaut suffisent. Il met les textes en minuscules et ne garde que les mots d'au moins deux caractères
- `vectorizer.fit_transform(donnees[:, 0])` : apprend le vocabulaire et les idf sur les paragraphes, puis renvoie `X`, une matrice creuse (`scipy.sparse.csr_matrix`) qui ne stocke que les valeurs non nulles
- `X.shape` : le nombre de lignes et de colonnes ; `X.nnz` : le nombre de valeurs non nulles
- pour vérifier : `X` a 48 333 lignes, 486 005 colonnes et 5 642 466 valeurs non nulles, soit 0,024 % des cases. Pleine, en `float64`, elle occuperait 188 Go ; creuse, elle en occupe 68 Mo

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
import sklearn.feature_extraction.text

vectorizer = sklearn.feature_extraction.text.TfidfVectorizer()
X = ...
print("Forme de X :", ...)
print("Valeurs non nulles :", ...)
print(f"Part non nulle : {...:.3%}")
```

### Solution

In [ ]:
import sklearn.feature_extraction.text

vectorizer = sklearn.feature_extraction.text.TfidfVectorizer()
X = vectorizer.fit_transform(donnees[:, 0])
print("Forme de X :", X.shape)
print("Valeurs non nulles :", X.nnz)
print(f"Part non nulle : {X.nnz / (X.shape[0] * X.shape[1]):.3%}")

## Un corpus bruité

Avant de chercher dans le corpus, regardez ce qu'il contient :

1. Combien de paragraphes compte chaque journal ? Qu'en attendre des résultats d'une recherche ?
2. Quelle part du vocabulaire de `vectorizer` n'apparaît que dans un seul paragraphe ? Ce sont les mots de plus grand idf (`vectorizer.idf_`).
3. Affichez 20 de ces mots, tirés au hasard : combien sont des mots bien écrits ? D'où viennent les autres ?

**Documentation utile**

- [`numpy.unique`](https://numpy.org/doc/stable/reference/generated/numpy.unique.html)
- [`sklearn.feature_extraction.text.TfidfVectorizer.get_feature_names_out`](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html#sklearn.feature_extraction.text.TfidfVectorizer.get_feature_names_out)
- [`numpy.random.Generator.choice`](https://numpy.org/doc/stable/reference/random/generated/numpy.random.Generator.choice.html)

In [ ]:
# Votre code ici

### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `numpy.unique(donnees[:, 1], return_counts=True)` : les journaux et, avec `return_counts=True`, le nombre de paragraphes de chacun
- `vectorizer.vocabulary_` : le vocabulaire, un dictionnaire qui associe à chaque mot sa colonne dans `X` ; `len(vectorizer.vocabulary_)` en donne la taille
- `vectorizer.idf_` : l'idf de chaque colonne, qui décroît avec le nombre de paragraphes contenant le mot. `vectorizer.idf_ == vectorizer.idf_.max()` est donc un tableau de booléens, vrai pour les mots d'un seul paragraphe, et sa moyenne (`.mean()`) la part cherchée
- `vectorizer.get_feature_names_out()` : les mots, dans l'ordre des colonnes ; indexés par ce tableau de booléens, il ne reste que les mots d'un seul paragraphe
- `numpy.random.default_rng(0).choice(mots, 20, replace=False)` : 20 mots tirés au hasard, sans remise ; la graine `0` donne les mêmes à chaque exécution
- pour vérifier : 41 998 paragraphes de *L'Humanité* et 6 335 du *Figaro* ; 360 882 des 486 005 mots du vocabulaire (74 %) n'apparaissent que dans un paragraphe

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
journaux, nb_paragraphes = ...
print(journaux, nb_paragraphes)

un_seul_paragraphe = ...  # vrai pour les mots d'un seul paragraphe
print(f"{un_seul_paragraphe.sum()} mots sur {...} : {...:.0%}")

mots = vectorizer.get_feature_names_out()
print(numpy.random.default_rng(0).choice(..., 20, replace=False))
```

### Solution

In [ ]:
journaux, nb_paragraphes = numpy.unique(donnees[:, 1], return_counts=True)
print(journaux, nb_paragraphes)

un_seul_paragraphe = vectorizer.idf_ == vectorizer.idf_.max()
print(
  f"{un_seul_paragraphe.sum()} mots sur {len(vectorizer.vocabulary_)} : "
  f"{un_seul_paragraphe.mean():.0%}"
)

mots = vectorizer.get_feature_names_out()
print(numpy.random.default_rng(0).choice(mots[un_seul_paragraphe], 20, replace=False))

Ce que montrent ces chiffres :

- *L'Humanité* fournit 87 % des paragraphes : la plupart des résultats d'une recherche en viendront, sans que *Le Figaro* parle moins du sujet. Il n'a d'ailleurs aucun numéro entre le 11 juin et le 7 août 1914.
- 74 % du vocabulaire n'apparaît que dans un paragraphe. Parmi les 20 mots tirés, la plupart sont des mots mal lus par l'OCR (`tiavaux` pour « travaux », `licbknedii` pour « Liebknecht »), les autres des nombres ou des mots rares (`maniabilité`).
- Ces mots ont le plus grand idf, celui que la diapositive « TF-IDF — Intérêt de $df_i$ » donne aux mots rares, les plus discriminants. Ici, ils ne discriminent rien : aucune requête ne les contient. Et un paragraphe dont l'OCR a mal lu un mot de la requête échappe à la recherche par ce mot.

## Similarité

Implémentez une fonction `best_score(requete: str) -> None` qui :

- applique à la requête le prétraitement des paragraphes (`pretraiter_texte`), puis la transforme en vecteur TF-IDF avec `vectorizer` ;
- calcule la similarité cosinus de chaque paragraphe de `X` avec la requête ;
- affiche les 5 paragraphes les plus similaires, du premier au cinquième : le journal, la date, la similarité, l'index du paragraphe dans `donnees` et son texte.

Testez-la avec `best_score("attentat sarajevo")` : de quel journal viennent les résultats, et pourquoi ? Retrouvez-vous l'attentat du 28 juin 1914 ? Quelle erreur d'OCR contient le titre qui annonce la mort de l'archiduc ?

**Documentation utile**

- [`sklearn.metrics.pairwise.cosine_similarity`](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.pairwise.cosine_similarity.html)
- [`numpy.argsort`](https://numpy.org/doc/stable/reference/generated/numpy.argsort.html)

In [ ]:
def best_score(requete: str) -> None:
  # Votre code ici
  ...


# Votre code ici

### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `vectorizer.transform([pretraiter_texte(requete)])` : le vecteur TF-IDF de la requête, une matrice d'une ligne. `transform`, et non `fit_transform` : le vocabulaire et les idf restent ceux du corpus. `TfidfVectorizer` mettrait déjà la requête en minuscules et ignorerait sa ponctuation ; `pretraiter_texte` la garde traitée comme les paragraphes si ce prétraitement change (pour retirer les accents, par exemple)
- `sklearn.metrics.pairwise.cosine_similarity(X, vecteur_requete)[:, 0]` : la similarité de chaque paragraphe avec la requête (une matrice d'une colonne, dont `[:, 0]` fait un tableau)
- `(-similarites).argsort()[:5]` : `argsort` donne les indices qui trient un tableau par ordre croissant ; sur l'opposé des similarités, les 5 premiers sont ceux des 5 plus grandes
- `texte, journal, date = donnees[index]` : les trois colonnes d'une ligne
- pour vérifier : sur `attentat sarajevo`, la meilleure similarité vaut 0,54

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
import sklearn.metrics.pairwise


def best_score(requete: str) -> None:
  vecteur_requete = ...
  similarites = ...
  for index in ...:  # les indices des 5 plus grandes similarités
    texte, journal, date = donnees[index]
    print(
      f"{journal:10} du {date} ; "
      f"similarité {similarites[index]:5.2f} ; "
      f"index {index:6} ; "
      f"{texte}"
    )


best_score("attentat sarajevo")
```

### Solution

In [ ]:
import sklearn.metrics.pairwise


def best_score(requete: str) -> None:
  # Le même prétraitement que les paragraphes : TfidfVectorizer mettrait déjà la
  # requête en minuscules, mais requête et paragraphes doivent rester traités pareil
  vecteur_requete = vectorizer.transform([pretraiter_texte(requete)])
  similarites = sklearn.metrics.pairwise.cosine_similarity(X, vecteur_requete)[:, 0]
  # argsort trie par ordre croissant : sur l'opposé des similarités, les indices des
  # plus grandes viennent en premier
  for index in (-similarites).argsort()[:5]:
    texte, journal, date = donnees[index]
    print(
      f"{journal:10} du {date} ; "
      f"similarité {similarites[index]:5.2f} ; "
      f"index {index:6} ; "
      f"{texte}"
    )


best_score("attentat sarajevo")

Les 5 résultats viennent de *L'Humanité*, qui fournit 87 % des paragraphes et seule couvre la fin juin. Le deuxième, du 29 juin, annonce la mort de l'archiduc héritier « d adtriche » : une recherche sur `autriche` ne le trouverait pas par ce mot.

## BM25

Essayez d'abord `best_score("guerre allemands")` : que remarquez-vous sur la longueur des paragraphes en tête ?

Le cours présente Okapi BM25, une amélioration du produit scalaire TF-IDF, qui compare la longueur de chaque document à la longueur moyenne :

$$
\text{score}(D, Q)
= \sum_{i = 1}^n
\text{IDF}(q_i)
\cdot
\frac{f(q_i, D) \cdot (k_1 + 1)}
     {f(q_i, D) + k_1 \cdot \left(1 - b + b \cdot \frac{|D|}{\bar{|D|}}\right)}
$$

où :

- $q_1, \dots, q_n$ sont les termes de la requête $Q$
- $\text{IDF}(q_i) = \log\frac{N}{df_i}$, celui de TF-IDF : $df_i$ documents sur les $N$ de la base contiennent $q_i$
- $f(q_i, D)$ est la fréquence du terme $q_i$ dans le document $D$
- $|D|$ est le nombre de termes de $D$, et $\bar{|D|}$ la longueur moyenne des documents de la base
- $k_1 = 1{,}5$ et $b = 0{,}75$

Ici, un document est un paragraphe. Comptez les termes de chaque paragraphe avec un `CountVectorizer`, dans une variable `compteur` et une matrice `comptes`, et gardez la longueur de chaque paragraphe dans `longueurs`. Implémentez ensuite la fonction `scores_bm25`, qui renvoie le score de chaque paragraphe pour une requête.

**Documentation utile**

- [`sklearn.feature_extraction.text.CountVectorizer`](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.CountVectorizer.html)
- [`scipy.sparse.csr_matrix.getnnz`](https://docs.scipy.org/doc/scipy/reference/generated/scipy.sparse.csr_matrix.getnnz.html)
- [`numpy.log`](https://numpy.org/doc/stable/reference/generated/numpy.log.html)

In [ ]:
# Votre code ici


def scores_bm25(requete: str, k1: float = 1.5, b: float = 0.75) -> numpy.ndarray:
  # Votre code ici
  ...

### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `compteur.fit_transform(donnees[:, 0])` : la matrice creuse `comptes`, avec une ligne par paragraphe et une colonne par terme
- `comptes.getnnz(axis=0)` : pour chaque terme, le nombre de paragraphes qui le contiennent ($df_i$)
- `numpy.asarray(comptes.sum(axis=1)).ravel()` : pour chaque paragraphe, son nombre de termes ($|D|$). `CountVectorizer` ignore les mots d'une lettre : 12 paragraphes n'ont donc aucun terme, comptez-les pour 1 (`numpy.maximum(..., 1)`), sinon $b = 1$ divise par zéro
- `compteur.transform([requete]).indices` : les colonnes des termes de la requête. `CountVectorizer` la met en minuscules, la découpe et ignore les mots absents du corpus
- `comptes[:, terme].toarray().ravel()` : la colonne d'un terme, soit $f(q_i, D)$ pour tous les paragraphes. Les opérations de numpy portent alors sur tous les paragraphes à la fois, sans boucle sur les documents
- pour vérifier : `N` vaut 48 333, `longueurs.mean()` environ 178 et `scores_bm25("jaurès assassiné").max()` environ 13,51

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
compteur = sklearn.feature_extraction.text.CountVectorizer()
comptes = ...
N = comptes.shape[0]
df = ...  # df_i pour chaque terme
longueurs = numpy.maximum(..., 1)  # |D| pour chaque paragraphe


def scores_bm25(requete: str, k1: float = 1.5, b: float = 0.75) -> numpy.ndarray:
  # La partie du dénominateur qui ne dépend que du paragraphe
  normalisation = k1 * (1 - b + b * ...)
  scores = numpy.zeros(N)
  for terme in compteur.transform([requete]).indices:
    f = ...  # f(q_i, D) pour tous les paragraphes
    idf = ...
    scores += ...
  return scores
```

### Solution

In [ ]:
best_score("guerre allemands")

compteur = sklearn.feature_extraction.text.CountVectorizer()
comptes = compteur.fit_transform(donnees[:, 0])
N = comptes.shape[0]
df = comptes.getnnz(axis=0)
# CountVectorizer ignore les mots d'une lettre : 12 paragraphes n'ont aucun terme, et on
# les compte pour 1 pour ne pas diviser par zéro quand b vaut 1
longueurs = numpy.maximum(numpy.asarray(comptes.sum(axis=1)).ravel(), 1)


def scores_bm25(requete: str, k1: float = 1.5, b: float = 0.75) -> numpy.ndarray:
  normalisation = k1 * (1 - b + b * longueurs / longueurs.mean())
  scores = numpy.zeros(N)
  for terme in compteur.transform([requete]).indices:
    f = comptes[:, terme].toarray().ravel()
    idf = numpy.log(N / df[terme])
    scores += idf * f * (k1 + 1) / (f + normalisation)
  return scores


print(scores_bm25("jaurès assassiné").max())

## Comparaison avec le cosinus

Implémentez une fonction `compare(requete: str) -> None` qui affiche, pour la similarité cosinus puis pour BM25, les 5 meilleurs résultats : le score, la longueur, l'index et le début du texte. Le texte entier d'un résultat se lit dans `donnees[index, 0]`.

Regardez ensuite :

1. sur `guerre allemands` et `jaurès assassiné`, la longueur des paragraphes que chaque méthode met en tête, par rapport à la longueur médiane du corpus. Laquelle remonte l'article sur l'assassinat de Jaurès ?
2. la longueur des 5 premiers résultats de BM25 sur `guerre allemands`, avec $b$ valant 0, puis 0,75, puis 1 : que règle $b$ ?
3. sur `mobilisation générale`, les résultats des deux méthodes : combien annoncent vraiment une mobilisation générale ? Pourquoi ?

Concluez : qu'apporte la normalisation par la longueur, et que ne résout-elle pas ?

**Documentation utile**

- [`numpy.argsort`](https://numpy.org/doc/stable/reference/generated/numpy.argsort.html)
- [`numpy.median`](https://numpy.org/doc/stable/reference/generated/numpy.median.html)

In [ ]:
def compare(requete: str) -> None:
  # Votre code ici
  ...


# Votre code ici

### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `(-scores).argsort()[:5]` : les indices des 5 meilleurs scores, comme dans `best_score`
- `sklearn.metrics.pairwise.cosine_similarity(X, vecteur_requete)[:, 0]` : la similarité de chaque paragraphe avec la requête
- `donnees[index, 0][:80]` : le début d'un paragraphe
- `scores_bm25("guerre allemands", b=b)` : BM25 avec une autre valeur de $b$
- pour vérifier : `numpy.median(longueurs)` vaut 102

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
def affiche_top5(scores: numpy.ndarray) -> None:
  for index in ...:  # les indices des 5 meilleurs scores
    debut = donnees[index, 0][:80]
    print(f"{scores[index]:5.2f} ; {longueurs[index]:4} termes ; {index:6} ; {debut}")


def compare(requete: str) -> None:
  print(f"Requête : {requete}")
  print("Cosinus")
  vecteur_requete = vectorizer.transform([requete])
  affiche_top5(...)  # la similarité de chaque paragraphe avec la requête
  print("BM25")
  affiche_top5(...)


print("Longueur médiane :", ...)
compare("guerre allemands")
compare("jaurès assassiné")

for b in (0, 0.75, 1):
  meilleurs = ...  # les indices des 5 meilleurs scores de BM25 avec ce b
  print(f"b = {b} : {longueurs[meilleurs]}")

compare("mobilisation générale")
```

### Solution

In [ ]:
def affiche_top5(scores: numpy.ndarray) -> None:
  for index in (-scores).argsort()[:5]:
    debut = donnees[index, 0][:80]
    print(f"{scores[index]:5.2f} ; {longueurs[index]:4} termes ; {index:6} ; {debut}")


def compare(requete: str) -> None:
  print(f"Requête : {requete}")
  print("Cosinus")
  vecteur_requete = vectorizer.transform([requete])
  affiche_top5(sklearn.metrics.pairwise.cosine_similarity(X, vecteur_requete)[:, 0])
  print("BM25")
  affiche_top5(scores_bm25(requete))


print("Longueur médiane :", numpy.median(longueurs))
compare("guerre allemands")
compare("jaurès assassiné")

for b in (0, 0.75, 1):
  meilleurs = (-scores_bm25("guerre allemands", b=b)).argsort()[:5]
  print(f"b = {b} : {longueurs[meilleurs]}")

compare("mobilisation générale")

Ce que montrent ces résultats :

- La similarité cosinus divise par la norme du vecteur du paragraphe : dans un titre, les termes de la requête pèsent lourd. Sur `guerre allemands`, ses 5 premiers résultats font 12 à 14 termes, pour une médiane de 102. Sur `jaurès assassiné`, elle met en tête un marchand de vins (10 termes) et l'article sur Jaurès (691 termes) n'arrive que 3e.
- BM25 met cet article en tête, et les quatre suivants parlent aussi de la mort de Jaurès. Sur `guerre allemands`, deux paragraphes de 84 et 85 termes entrent dans les 5 premiers, mais trois titres restent : les textes courts sont moins favorisés, ils ne sont pas écartés.
- $b$ règle cette normalisation : à 0, les 5 premiers font 656 à 1 042 termes, car un long texte contient plus souvent les termes ; à 1, BM25 retrouve des titres de 12 à 14 termes, comme le cosinus.
- La normalisation ne règle pas tout : sur `mobilisation générale`, un seul résultat sur 5 l'annonce vraiment (celle de la Hollande), dans les deux méthodes. Une grève générale en Afrique du Sud, avec une mobilisation de troupes, arrive 1re au cosinus et 2e avec BM25 : un sac de mots ne sait pas que les deux termes doivent se suivre. Le récit de l'affichage de l'ordre à Paris n'arrive que 6e avec BM25, 16e au cosinus.

Trois requêtes lues à la main ne mesurent rien : pour comparer deux moteurs, il faut des jugements de pertinence.